# **Author**: Mukhamedali Daniyaruly
# **Topic**: Training C Abstractor layer in Vision KazLM model on kaz-coco dataset
# **Date**: 22/01/2026

In [ ]:
!pip install -U bitsandbytes transformers

In [ ]:
import os
import math
import random
import glob
import json
from tqdm.auto import tqdm
from google.colab import drive
from PIL import Image

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn
import wandb

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import AdamW
from torch.utils.data import Dataset, DataLoader, random_split

from transformers import AutoModelForCausalLM, AutoTokenizer, AutoModel, AutoProcessor, BitsAndBytesConfig, get_cosine_schedule_with_warmup

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
device

In [ ]:
drive.mount("/content/drive")

In [ ]:
json_path = "/content/drive/MyDrive/Vision KazLM/COCO_Data"
save_path = "/content/drive/MyDrive/Vision KazLM/3types/c abstractor/training_checkpoint"

# Configuration

In [ ]:
drive.mount("/content/drive")
torch.manual_seed(42)
class Configuration:
  def __init__(self):
    # Paths
    self.data_path = "/content/drive/MyDrive/Vision KazLM/COCO_Data/coco_kazakh_train_FULL.json"
    self.images_path = "/content/drive/MyDrive/Vision KazLM/COCO_Data/images"
    self.save_path = "/content/drive/MyDrive/Vision KazLM/3types/c abstractor/training_checkpoint"

    # Models
    self.llm_model_name = "issai/LLama-3.1-KazLLM-1.0-8B"
    self.vision_model_name = "google/siglip-so400m-patch14-384"

    # Hyperparameters
    self.vision_embed_dim = 1152
    self.llm_embed_dim = 4096
    self.max_length = 128
    self.epochs = 1
    self.lr = 2e-5
    self.max_length = 128
    self.batch_size = 2
    self.grad_accumulation = 8
    self.log_steps = 2
    self.warmup_ratio = 0.05
    self.weight_decay = 0.01

    # device
    self.device = "cuda" if torch.cuda.is_available() else "cpu"

config = Configuration()
os.makedirs(config.save_path, exist_ok=True)

# Download images

In [ ]:
import os

# Create a structure
root_dir = "/content/coco_karpathy"
os.makedirs(f"{root_dir}/images", exist_ok=True)
os.makedirs(f"{root_dir}/annotations", exist_ok=True)

# Download images 2014 (Train + Val) ~13GB + 6GB
!wget -c http://images.cocodataset.org/zips/train2014.zip -O {root_dir}/train2014.zip
!unzip -q {root_dir}/train2014.zip -d {root_dir}/images/
!rm {root_dir}/train2014.zip

!wget -c http://images.cocodataset.org/zips/val2014.zip -O {root_dir}/val2014.zip
!unzip -q {root_dir}/val2014.zip -d {root_dir}/images/
!rm {root_dir}/val2014.zip

# Dataset Class

In [ ]:
class VLMDataset(Dataset):
    def __init__(self, json_path, images_root, tokenizer, processor, max_length=128):
        self.images_root = images_root
        self.tokenizer = tokenizer
        self.processor = processor
        self.max_length = max_length

        print(f"Loading data from {json_path}...")
        with open(json_path, 'r', encoding='utf-8') as f:
            self.data = json.load(f)
        print(f"Loaded {len(self.data)} samples")

        # Filter out entries with empty captions_kk to prevent IndexError
        initial_len = len(self.data)
        self.data = [entry for entry in self.data if entry.get('captions_kk')]
        if len(self.data) < initial_len:
            print(f"Filtered out {initial_len - len(self.data)} samples with empty 'captions_kk'. New dataset size: {len(self.data)}")


    def __len__(self):

      # Return Length of dataset
        return len(self.data)

    def __getitem__(self, idx):
        entry = self.data[idx]

        # 1. Load Image
        image_path = os.path.join(self.images_root, entry['image_path'])
        try:
            image = Image.open(image_path).convert('RGB')
        except:
            # If image loading fails, try a different valid item
            return self.__getitem__(random.randint(0, len(self.data)-1)) # Fallback

        # 2. Process Image (SigLIP)
        # pixel_values: [3, 384, 384]
        pixel_values = self.processor(images=image, return_tensors="pt").pixel_values.squeeze(0)

        # 3. Process Text
        # Instruction: "Суреттегі көрініс: [CAPTION]"
        # Ensure captions_kk is not empty before calling random.choice
        if not entry['captions_kk']:
            # This case should ideally be caught by filtering during init, but as a safeguard
            return self.__getitem__(random.randint(0, len(self.data)-1)) # Fallback to another item
        caption = random.choice(entry['captions_kk'])
        prompt = "Суреттегі көрініс: "
        full_text = prompt + caption + self.tokenizer.eos_token

        # Tokenize
        tokenized = self.tokenizer(
            full_text,
            max_length=self.max_length,
            padding="max_length",
            truncation=True,
            return_tensors="pt"
        )

        input_ids = tokenized.input_ids.squeeze(0)
        attention_mask = tokenized.attention_mask.squeeze(0)

        # 4. Create Labels (Masking user prompt)
        # We want to model train predict caption, not instruction
        labels = input_ids.clone()

        # Находим длину промпта в токенах, чтобы скрыть его
        prompt_len = len(self.tokenizer(prompt).input_ids) - 1 # -1 for deleting bos token
        labels[:prompt_len] = -100 # Mask prompt
        labels[labels == self.tokenizer.pad_token_id] = -100 # Mask padding

        return {
            "pixel_values": pixel_values,
            "input_ids": input_ids,
            "attention_mask": attention_mask,
            "labels": labels
        }

In [ ]:
def collate_fn(batch):
    # Batch это список словарей: [dict1, dict2, ...]
    # Нам нужно сделать словарь списков тензоров: {key: stack([t1, t2])}

    pixel_values = torch.stack([item['pixel_values'] for item in batch])
    input_ids = torch.stack([item['input_ids'] for item in batch])
    attention_mask = torch.stack([item['attention_mask'] for item in batch])
    labels = torch.stack([item['labels'] for item in batch])

    return {
        "pixel_values": pixel_values,
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "labels": labels
    }

In [ ]:
images_path = os.path.join("coco_karpathy", "images")

In [ ]:
tokenizer.pad_token_id = tokenizer.eos_token_id
full_dataset = VLMDataset(
    json_path=config.data_path, # Path to json file
    images_root=images_path, # Images path
    tokenizer=tokenizer, # Tokenizer
    processor=processor,   # Processor

)

In [ ]:
train_size = int(0.9 * len(full_dataset))
val_size = int(0.05 * len(full_dataset))
test_size = len(full_dataset) - train_size - val_size

train_dataset, val_dataset, test_dataset = random_split(
    full_dataset, [train_size, val_size, test_size]
)

In [ ]:
train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=8, shuffle=False, num_workers=2, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=8, shuffle=False, num_workers=2, pin_memory=True)

In [ ]:
for batch in train_loader:
  px, ins, masks, lbs = batch["pixel_values"], batch["input_ids"], batch["attention_mask"], batch["labels"]

  print(px.shape)
  print(ins.shape)
  print(masks.shape)
  print(lbs.shape)

  break

# Model Architecture

In [ ]:
class VisionEncoder(nn.Module):
  def __init__(self, vision_encoder):
    super().__init__()

    self.vision_encoder = vision_encoder

    for p in self.vision_encoder.parameters():
      p.requires_grad=False

  def forward(self, pixel_values):
    outs = self.vision_encoder.vision_model(pixel_values=pixel_values)
    return outs.last_hidden_state

In [ ]:
class CAbstractor(nn.Module):
    def __init__(self, vision_embeds, llm_embeds, mlp_ratio=4):
        super().__init__()

        self.input_dim = vision_embeds * 4
        self.llm_embeds = llm_embeds

        # Hidden size for MLP
        hidden_size = int(llm_embeds * mlp_ratio)

        # Layers pass this dimension (4608)
        self.up = nn.Linear(self.input_dim, hidden_size)
        self.gate = nn.Linear(self.input_dim, hidden_size)
        self.down = nn.Linear(hidden_size, llm_embeds)

        # use SiLU (LLaVA style)
        self.act = nn.SiLU()

    def forward(self, images):
        # images: [Batch, 729, 1152]
        B, L, D = images.size()

        h = w = int(L ** 0.5) # 27
        new_h = new_w = (h - 1) // 2 # 13

        # --- Spatial Downsampling Logic ---
        # 1. Восстанавливаем 2D картинку
        images = images.view(B, h, w, D)
        # 2. Отрезаем последний пиксель, чтобы стало четным (26x26)
        images = images[:, :h-1, :w-1, :]
        # 3. Pixel Shuffle (Space to Depth): 26x26 -> 13x13x4
        images = images.reshape(B, new_h, new_w, 2, 2, D)
        # 4. Flatten channels: [Batch, 13, 13, 4608] -> [Batch, 169, 4608]
        images = images.view(B, new_h * new_w, 4 * D)

        # --- MLP Projection ---
        # Теперь размерности совпадают
        return self.down(self.up(images) * self.act(self.gate(images)))

In [ ]:
class KazVLM(nn.Module):
    def __init__(self, vision_model, llm_model):
        super().__init__()
        self.vision_encoder = VisionEncoder(vision_model)
        self.llm = llm_model

        vision_dim = 1152
        llm_dim = 4096

        self.abstractor = CAbstractor(
            vision_embeds=vision_dim,
            llm_embeds=llm_dim,
            mlp_ratio=4
        )

        # Заморозка LLM
        for param in self.llm.parameters():
            param.requires_grad = False

        # === [NEW 1/2] Включаем Gradient Checkpointing в самой модели ===
        # Это позволяет не хранить промежуточные активации, экономя VRAM
        self.llm.gradient_checkpointing_enable()

        trainable_params = sum(p.numel() for p in self.abstractor.parameters())
        print(f"Model Initialized. Trainable Params (Abstractor): {trainable_params}")

    def forward(self, pixel_values, input_ids, attention_mask, labels=None):

        with torch.no_grad():
            image_features = self.vision_encoder(pixel_values)

        image_embeds = self.abstractor(image_features)
        text_embeds = self.llm.get_input_embeddings()(input_ids)
        inputs_embeds = torch.cat([image_embeds, text_embeds], dim=1)

        batch_size = input_ids.shape[0]
        image_mask = torch.ones((batch_size, image_embeds.shape[1]), device=image_embeds.device)
        full_attention_mask = torch.cat([image_mask, attention_mask], dim=1)

        if labels is not None:
            image_labels = torch.full((batch_size, image_embeds.shape[1]), -100, device=labels.device)
            full_labels = torch.cat([image_labels, labels], dim=1)
        else:
            full_labels = None

        outputs = self.llm(
            inputs_embeds=inputs_embeds,
            attention_mask=full_attention_mask,
            labels=full_labels,
            # === [NEW 2/2] Отключаем кэш при чекпоинтинге ===
            # Gradient Checkpointing не совместим с use_cache=True
            use_cache=False
        )

        return outputs

# Download models: SigLIP + KazLM

In [ ]:
# We need to verify us to get access for using KazLM model
from huggingface_hub import login
login("YOUR_HUGGINGFACE_TOKEN")

In [ ]:
# Quantization
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True
)

In [ ]:
# Download tokenizer and set padding token
tokenizer = AutoTokenizer.from_pretrained(config.llm_model_name)
tokenizer_pad_token = tokenizer.eos_token

# Download processor
processor = AutoProcessor.from_pretrained(config.vision_model_name)

In [ ]:
# Download SiLIP model and set to cuda
vision_model = AutoModel.from_pretrained(
    config.vision_model_name,
    torch_dtype=torch.bfloat16,
    device_map="cuda"
)

In [ ]:
# Download KazLM and Quantize it
llm_model = AutoModelForCausalLM.from_pretrained(config.llm_model_name, quantization_config=bnb_config, device_map="cuda")

# Training Configuration

In [ ]:
# Initialize a Vision Language Model
model = KazVLM(vision_model, llm_model)

In [ ]:
model = model.to(device)

In [ ]:
# optimizer
num_training_steps = int((1 * config.epochs) // config.grad_accumulation)
num_warmup_steps = int(num_training_steps * config.warmup_ratio)

scaler = torch.cuda.amp.GradScaler()
optimizer = torch.optim.AdamW(model.abstractor.parameters(), lr=config.lr)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=config.epochs * len(train_loader))

In [ ]:
# bismillah
for epoch in range(config.epochs):

  # set to training
  model.train()

  pbar = tqdm(train_loader)
  total_loss = 0

  # iterate train loader
  for step, batch in enumerate(pbar):
    pixel_values = batch["pixel_values"].to(device, dtype=torch.bfloat16)
    input_ids = batch["input_ids"].to(device)
    attention_mask = batch["attention_mask"].to(device)
    labels = batch["labels"].to(device)

    # Use mixed precision
    with torch.amp.autocast(device_type=device, dtype=torch.bfloat16):
      outputs = model(pixel_values, input_ids, attention_mask, labels)
      loss = outputs.loss
      loss = loss / config.grad_accumulation

    scaler.scale(loss).backward()

    # Use gradient accumulation
    if (step + 1) % config.grad_accumulation == 0:
      scaler.unscale_(optimizer)
      torch.nn.utils.clip_grad_norm_(model.projector.parameters(), 1.0)

      scaler.step(optimizer)
      scaler.update()
      scheduler.step()
      optimizer.zero_grad()

    total_loss += loss.item()
    pbar.set_description(f"Epoch: {epoch + 1} | Loss: {loss.item() * config.grad_accumulation:.4f} | PPL: {math.exp(loss.item() * config.grad_accumulation):.4f}")

    # save each 500 step
    if (step + 1) % 500 == 0:
      torch.save({
          "epoch": {epoch + 1},
          "step": {step},
          "model_abstractor_state_dict": model.abstractor.state_dict(),
      }, f"{config.save_path}/abstractor_step_{step}.pth")

  # Logging on validation dataset
  if (epoch + 1) % config.log_steps == 0: # Changed config.log_step to config.log_steps

    # set to evaluation state in validation part
    model.eval()

    val_total_loss = 0

    with torch.no_grad():

    # iterate val loader
      for batch in tqdm(val_loader, desc="Validation"):

        pixel_values = batch["pixel_values"].to(device, dtype=torch.bfloat16)
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device) # Changed `batch = batch["labels"].to(device)` to `labels = batch["labels"].to(device)`

        # Use mixed precision
        with torch.amp.autocast(device_type=device, dtype=torch.bfloat16):
          outputs = model(pixel_values, input_ids, attention_mask, labels)
          val_loss = outputs.loss

        # compute validation loss
        val_total_loss += val_loss.item()

      avg_val_loss = val_total_loss / len(val_loader)
      avg_val_ppl = math.exp(avg_val_loss)

      print(f"Epoch: {epoch + 1} | Val_Loss: {avg_val_loss} | Val_PPL: {avg_val_ppl}")

  # Compute training loss
  avg_loss = total_loss / len(train_loader)
  avg_ppl = math.exp(avg_loss)
  print("-----" * 20)
  print(f"Epoch: {epoch + 1} | Loss: {avg_loss} | PPL: {avg_ppl}")

  # Save each epoch
  torch.save({
      "epoch": {epoch + 1},
      "model_abstractor_state_dict": model.projector.state_dict(),
      "optimizer_state_dict": optimizer.state_dict(),
      "scaler_state_dict": scaler.state_dict(),
      "loss": avg_loss,
      "ppl": avg_ppl,
      "val_loss": avg_val_loss,
      "val_ppl": avg_val_ppl
  }, f"{config.save_path}/abstractor_epoch_{epoch+1}.pth")

In [ ]:
global_val_loss = []
global_ppl_loss = []

In [ ]:
# bismillah
for epoch in range(config.epochs):
    model.train()

    # Инициализируем progress bar
    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{config.epochs}")
    total_loss = 0

    for step, batch in enumerate(pbar):
        # Перенос на устройство
        pixel_values = batch["pixel_values"].to(device, dtype=torch.bfloat16)
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        # Autocast (device_type должен быть строкой 'cuda')
        with torch.amp.autocast(device_type="cuda", dtype=torch.bfloat16):
            outputs = model(pixel_values, input_ids, attention_mask, labels)
            loss = outputs.loss
            # Scale loss for accumulation
            loss = loss / config.grad_accumulation

        # Backward
        scaler.scale(loss).backward()

        # Gradient Accumulation Step
        if (step + 1) % config.grad_accumulation == 0:
            scaler.unscale_(optimizer)

            # ВАЖНО: Клипаем градиенты у ВСЕХ обучаемых параметров (и Abstractor, и LoRA)
            # Фильтруем те, у которых есть градиент
            trainable_params = [p for p in model.parameters() if p.requires_grad]
            torch.nn.utils.clip_grad_norm_(trainable_params, 1.0)

            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            optimizer.zero_grad()

        # Logging
        # Возвращаем лосс к реальному масштабу для статистики
        current_loss = loss.item() * config.grad_accumulation
        total_loss += current_loss

        pbar.set_description(f"Epoch: {epoch + 1} | Loss: {current_loss:.4f} | PPL: {math.exp(min(current_loss, 20)):.4f}")

        # Save checkpoint (every 500 steps)
        if (step + 1) % 500 == 0:
            torch.save({
                "epoch": epoch + 1, # Убрали фигурные скобки
                "step": step,
                "model_state_dict": model.abstractor.state_dict(), # Сохраняем все обучаемое
                "optimizer_state_dict": optimizer.state_dict(),
            }, f"{config.save_path}/checkpoint_step_{step}.pth")

    # --- Validation ---
    avg_val_loss = 0 # Инициализируем заранее
    avg_val_ppl = 0

    if (epoch + 1) % config.log_steps == 0:
        model.eval()
        val_total_loss = 0

        with torch.no_grad():
            for batch in tqdm(val_loader, desc="Validation"):
                pixel_values = batch["pixel_values"].to(device, dtype=torch.bfloat16)
                input_ids = batch["input_ids"].to(device)
                attention_mask = batch["attention_mask"].to(device)
                labels = batch["labels"].to(device)

                with torch.amp.autocast(device_type="cuda", dtype=torch.bfloat16):
                    outputs = model(pixel_values, input_ids, attention_mask, labels)
                    val_loss = outputs.loss

                val_total_loss += val_loss.item()

        avg_val_loss = val_total_loss / len(val_loader)
        avg_val_ppl = math.exp(min(avg_val_loss, 20)) # min защита от overflow
        print(f"Epoch: {epoch + 1} | Val_Loss: {avg_val_loss:.4f} | Val_PPL: {avg_val_ppl:.4f}")

        global_val_loss.append(avg_val_loss)
        global_ppl_loss.append(avg_val_ppl)


    # --- Epoch End Stats ---
    avg_loss = total_loss / len(train_loader)
    avg_ppl = math.exp(min(avg_loss, 20))

    print("-----" * 20)
    print(f"Epoch: {epoch + 1} Done | Train Loss: {avg_loss:.4f} | Train PPL: {avg_ppl:.4f}")

    # Save Epoch Checkpoint
    torch.save({
        "epoch": epoch + 1,
        "model_state_dict": model.abstractor.state_dict(), # Универсальное сохранение
        "optimizer_state_dict": optimizer.state_dict(),
        "scaler_state_dict": scaler.state_dict(),
        "loss": avg_loss,
        "val_loss": avg_val_loss
    }, f"{config.save_path}/model_epoch_{epoch+1}.pth")

# Evaluate

In [ ]:
model.eval()

losses = 0

for batch in test_loader:
  with torch.no_grad():
    pixel_values, input_ids, attention_mask, labels = [b.to(device) for batch]

    with torch.amp.autocast(torch.bfloat16):
      outs = model(pixel_values, input_ids, attention_mask, labels)

    loss += outs.loss.item()

print(f"Loss: {loss}, ppl: {math.exp(loss)}")

# Inference

In [ ]:
def inference_check(image, prompt="Суретте не көріп тұрсың?"):
  try:
    image = Image.open(image).convert("RGB")
  except Exception as e:
    print(f"Error at loading image {image}: {e}")

  pixel_values = processor(images=[image], return_tensors="pt").squeeze(0)
  image_features = vision_model(**pixel_values)
  image_features = model.linear_projection(image_features)

  text_inputs = tokenizer(
      prompt,
      max_length=128,
      truncation=True,
      padding="max_length",
      return_tensors="pt"
  )

  input_ids = text_inputs.input_ids
  attention_mask = text_inputs.attention_mask

  labels = input_ids.clone()
  labels[labels == tokenizer.pad_token_id] = -100

  outs = model(pixel_values, text=input_ids, attention_mask=attention_mask, labels=labels)

  return tokenizer.decode(outs[0][outs["input_ids"].shape[-1]:])